# JEPA-TTS Supercomputer Pipeline
This notebook automates the entire pipeline: downloading the fixed codebase, installing dependencies, training from scratch, and running inference/testing on the supercomputer.

### 1. Clone Codebase & Checkout Prototype Branch

In [ ]:
!git clone https://github.com/OmarA32/Audio-JEPA-Arabic-TTS.git
%cd Audio-JEPA-Arabic-TTS
!git checkout prototype/v5.0.0


### 2. Install Dependencies
Installs all libraries directly into the notebook kernel (including `vocos`).

In [ ]:
!pip install -r requirements.txt

### 3. Clear Old Weights (Safety)

In [ ]:
!rm -rf training_logs

### 4. Hugging Face Authentication (Optional)
If you want to automatically upload weights to Hugging Face during training, paste your Write token below. This saves it securely for the training script to use.

In [ ]:
import json

HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxx" # Replace with your real write token!

if HF_TOKEN.startswith("hf_") and len(HF_TOKEN) > 10:
    with open("hf_config.json", "w") as f:
        json.dump({"HF_TOKEN": HF_TOKEN}, f)
    print("Token saved! The training script will now automatically upload checkpoints.")
else:
    print("No valid token provided. Auto-upload disabled.")


### 4A. Run Training (From Scratch)\nThis deletes any existing weights and trains from scratch at Epoch 0. The dataset (`MohamedRashad/common-voice-18-arabic`) is public and will download automatically during the first epoch.

In [ ]:
# Training Arguments:
# --lang [arabic/english]       Language to train on.
# --db [nawar_halabi/ljspeech]  Database to use.
# --resume                      Add this to resume from latest checkpoint.
# --checkpointnum 5             Upload to Hugging Face every 5 epochs.
!python train.py --checkpointnum 5


### 4B. Resume Training / Fine-Tune\nIf you downloaded your saved weights from Hugging Face into the `training_logs` folder, run this cell instead! It will seamlessly resume training from the latest epoch.

In [ ]:
# Training Arguments:
# --lang [arabic/english]       Language to train on.
# --db [nawar_halabi/ljspeech]  Database to use.
# --resume                      Add this to resume from latest checkpoint.
# --checkpointnum 5             Upload to Hugging Face every 5 epochs.
!python train.py --resume --checkpointnum 5


### 5. Generate Inference Audio
Once training finishes (or if you manually stop the cell above after some epochs), run this cell to generate audio from text using your newly trained model.

In [ ]:
!python inference.py

### 6. Test Vocoder (Ground Truth Quality)
If you want to test the raw quality of the vocoder against the dataset (without the neural network's influence), run this test.

In [ ]:
!python test_vocoder_ground_truth.py --vocoder vocos

### 7. Test TTS model with new text.
You can pass any custom text to generate here:

In [ ]:
# Inference Arguments:
# --text      The text you want to synthesize.
# --output    Output WAV file name.
# --lang      [arabic/english] Language to use.
# --db        [nawar_halabi/ljspeech] Database the model trained on.
# --index     Optionally fetch text directly from the test dataset by index.
# --vocoder   [bigvgan/vocos] Vocoder to use.
!python inference.py --text "أي نص عربي تريد" --output "my_custom_audio.wav"

## ⬇️ Download Pre-Trained Weights
Use these cells to download your best models from Hugging Face back into your supercomputer when you start a new session.

### ⬇️ Download Arabic Model
Downloads the specific `.ckpt` file you specify from your Hugging Face repository.

In [ ]:
!pip install huggingface_hub

from huggingface_hub import hf_hub_download
import os

ARABIC_REPO = "KASP-JEPA/Project"
ARABIC_FILENAME = "best-epoch=000.ckpt" # <-- Type exactly which file you want to download here!

print(f"Downloading {ARABIC_FILENAME}...")
os.makedirs("training_logs/arabic/nawar_halabi", exist_ok=True)
hf_hub_download(
    repo_id=ARABIC_REPO,
    filename=ARABIC_FILENAME,
    local_dir="training_logs/arabic/nawar_halabi"
)
print("Arabic model downloaded successfully!")


### ⬇️ Download English Model
Same as above, but for the English model.

In [ ]:
# from huggingface_hub import hf_hub_download
# import os

# ENGLISH_REPO = "KASP-JEPA/Project-English"
# ENGLISH_FILENAME = "best-epoch=000.ckpt" # <-- Type exactly which file you want to download here!

# print(f"Downloading {ENGLISH_FILENAME}...")
# os.makedirs("training_logs/english/ljspeech", exist_ok=True)
# hf_hub_download(
#     repo_id=ENGLISH_REPO,
#     filename=ENGLISH_FILENAME,
#     local_dir="training_logs/english/ljspeech"
# )
# print("English model downloaded successfully!")


## ⬆️ Model Publishing

In [ ]:
# 🚀 Optional: Upload Best Models to Hugging Face
# You can upload your best trained models to the Hugging Face Hub so you can download them later!
# Get your Hugging Face write token from: https://huggingface.co/settings/tokens
!pip install huggingface_hub

HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxx" # Replace with your real token!

# --- 🟢 UPLOAD ARABIC MODEL ---
ARABIC_REPO = "KASP-JEPA/Project" 
ARABIC_CKPT = "training_logs/arabic/nawar_halabi/best-epoch=000.ckpt" # Change 000 to your best Arabic epoch!

!python upload_to_hf.py --ckpt {ARABIC_CKPT} --repo {ARABIC_REPO} --token {HF_TOKEN}

# --- 🔵 UPLOAD ENGLISH MODEL (Uncomment when ready) ---
# ENGLISH_REPO = "KASP-JEPA/Project-English"
# ENGLISH_CKPT = "training_logs/english/ljspeech/best-epoch=000.ckpt" # Change 000 to your best English epoch!

# !python upload_to_hf.py --ckpt {ENGLISH_CKPT} --repo {ENGLISH_REPO} --token {HF_TOKEN}


## ⬇️ Download Pre-Trained Weights
Use these cells to download your best models from Hugging Face back into your supercomputer when you start a new session.

### ⬇️ Download Arabic Model
Downloads the specific `.ckpt` file you specify from your Hugging Face repository.

In [ ]:
!pip install huggingface_hub

from huggingface_hub import hf_hub_download
import os

ARABIC_REPO = "KASP-JEPA/Project-Arabic"
ARABIC_FILENAME = "best-epoch=000.ckpt" # <-- Type exactly which file you want to download here!

print(f"Downloading {ARABIC_FILENAME}...")
os.makedirs("training_logs/arabic/nawar_halabi", exist_ok=True)
hf_hub_download(
    repo_id=ARABIC_REPO,
    filename=ARABIC_FILENAME,
    local_dir="training_logs/arabic/nawar_halabi"
)
print("Arabic model downloaded successfully!")


### ⬇️ Download English Model
Same as above, but for the English model.

In [ ]:
# from huggingface_hub import hf_hub_download
# import os

# ENGLISH_REPO = "KASP-JEPA/Project-English"
# ENGLISH_FILENAME = "best-epoch=000.ckpt" # <-- Type exactly which file you want to download here!

# print(f"Downloading {ENGLISH_FILENAME}...")
# os.makedirs("training_logs/english/ljspeech", exist_ok=True)
# hf_hub_download(
#     repo_id=ENGLISH_REPO,
#     filename=ENGLISH_FILENAME,
#     local_dir="training_logs/english/ljspeech"
# )
# print("English model downloaded successfully!")


## ⬆️ Model Publishing

In [ ]:
# 🚀 Optional: Upload Best Models to Hugging Face
# You can upload your best trained models to the Hugging Face Hub so you can download them later!
# Get your Hugging Face write token from: https://huggingface.co/settings/tokens
!pip install huggingface_hub

HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxx" # Replace with your real token!

# --- 🟢 UPLOAD ARABIC MODEL ---
ARABIC_REPO = "KASP-JEPA/Project-Arabic" 
ARABIC_CKPT = "training_logs/arabic/nawar_halabi/best-epoch=000.ckpt" # Change 000 to your best Arabic epoch!

!python upload_to_hf.py --ckpt {ARABIC_CKPT} --repo {ARABIC_REPO} --token {HF_TOKEN}

# --- 🔵 UPLOAD ENGLISH MODEL (Uncomment when ready) ---
# ENGLISH_REPO = "KASP-JEPA/Project-English"
# ENGLISH_CKPT = "training_logs/english/ljspeech/best-epoch=000.ckpt" # Change 000 to your best English epoch!

# !python upload_to_hf.py --ckpt {ENGLISH_CKPT} --repo {ENGLISH_REPO} --token {HF_TOKEN}
